# Assignment 4 — Exercise 1

In [25]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.stats import norm


SEED = 42
plt.style.use("seaborn-v0_8")

calibrated_params = pd.read_json("SABR_calibrated_params.json")
calibrated_params

,calibrated_value
sigma0,0.294404
beta,1.000000
rho0,-1.000000
alpha0,0.405636
a,0.000001
b,0.091122


In [26]:
S0 = 100
K = 98
r = 0
q = 0.02
Ts = np.array([0.0, 0.25, 0.5, 0.75, 1])
deltat = 0.25

### Constant volatility

In the following part, we apply a constant volatility equal to 23%. We build the paths by simulating the stock price paths only at the 4 exercise dates.

In [27]:
# Constant volatility
sigma = 0.23
N = 1_000_000

exp = lambda z: np.exp((r-q-sigma**2 / 2) * deltat + sigma * np.sqrt(deltat) * z)

# Simulating a N * 4 matrix for S (N simulated paths, GBM)
S = np.full((N, len(Ts)), np.nan)
S[:, 0] = S0
for i in range(1, len(Ts)):
    z = norm.rvs(loc=0, scale=1, size=N, random_state=SEED)
    S[:, i] = S[:, i-1] * exp(z)
S = np.delete(S, [0], axis=1)

# Deriving the A matrix for each path
A = np.cumsum(S, axis=1) / np.arange(1, S.shape[1] + 1)

A.shape, S.shape

((1000000, 4), (1000000, 4))

In [28]:
# Immediate payoff matrix
P = np.maximum(A - K, 0.0)

# Initialize the cashflow
cashflow = P[:, -1].copy()
t_idx = np.full(len(S), S.shape[1] - 1, dtype=int)
times = Ts[1:]

for k in np.arange(0, S.shape[1]-1, 1)[::-1]:
    itm = P[:, k] > 0

    s = S[itm, k]
    a = A[itm, k]

    X = np.column_stack([np.ones_like(s), s, s**2, s**3, a, a**2, a**3])
    Y = cashflow[itm] * np.exp(-r * (times[t_idx[itm]] - times[k]))

    beta, _, _, _ = np.linalg.lstsq(X, Y)
    C_hat = X @ beta

    exercise = P[itm, k] > C_hat
    idx = np.where(itm)[0][exercise]
    cashflow[idx] = P[idx, k]
    t_idx[idx] = k


pv = cashflow * np.exp(-r * times[t_idx])
price = pv.mean()
se = cashflow.std(ddof=1) / np.sqrt(N)

print(f"Price of the option is {price.round(3)} and its standard error is {se.round(3)}\nover {N} simulated paths")

Price of the option is 13.654 and its standard error is 0.023
over 1000000 simulated paths


### SABR stochastic volatility model

We now use the calibrated SABR model from assignment 3. The parameters were imported in a JSON file and displayed in the first cell. We first build paths by simulating the stock price at shorter intervals (weekly) and then use an Euler scheme for the simulations.

In [ ]:
# Load the calibrated SABR parameters and initialize the weekly time grid
params = calibrated_params["calibrated_value"].to_dict()
rng = np.random.default_rng(SEED)
steps_per_year = 52
dt = 1 / steps_per_year
exercise_times = Ts[1:]
exercise_steps = np.rint(exercise_times * steps_per_year).astype(int)
forward = np.full(N, float(S0))
volatility = np.full(N, params["sigma0"])
sabr_spots = np.empty((N, exercise_times.size))
previous_step = 0

In [ ]:
# Define the model and the random variables for the Brownian motions
for column, (exercise_step, exercise_time) in enumerate(zip(exercise_steps, exercise_times)):
    for step in range(previous_step, exercise_step):
        t = step * dt
        rho_t = params["rho0"] * np.exp(-params["a"] * t)
        alpha_t = params["alpha0"] * np.exp(-params["b"] * t)
        z_forward = rng.standard_normal(N)
        z_independent = rng.standard_normal(N)
        z_volatility = rho_t * z_forward + np.sqrt(1 - rho_t**2) * z_independent

        # Both Euler increments use the volatility at the start of the week
        alive = forward > 0
        forward[alive] += (volatility[alive] * forward[alive] ** params["beta"] * np.sqrt(dt) * z_forward[alive])
        np.maximum(forward, 0.0, out=forward)
        volatility *= 1 + alpha_t * np.sqrt(dt) * z_volatility

    # Convert the simulated forward to spot for the exercise payoff
    sabr_spots[:, column] = forward * np.exp((r - q) * exercise_time)
    previous_step = exercise_step

In [ ]:
# Average the observed exercise-date spots and compute the payoff at each date
sabr_averages = np.cumsum(sabr_spots, axis=1) / np.arange(1, exercise_times.size + 1)
sabr_payoffs = np.maximum(sabr_averages - K, 0.0)
sabr_cashflows = sabr_payoffs[:, -1].copy()
sabr_exercise_indices = np.full(N, exercise_times.size - 1, dtype=int)

In [ ]:
# Work backward through the exercise dates to estimate the value of waiting
for k in range(exercise_times.size - 2, -1, -1):
    itm = np.flatnonzero(sabr_payoffs[:, k] > 0)
    if itm.size == 0:
        continue

    spot = sabr_spots[itm, k] / S0
    average = sabr_averages[itm, k] / S0
    basis = np.column_stack(
        [np.ones_like(spot), spot, spot**2, spot**3, average, average**2, average**3]
    )
    
    # Discount each future cash flow back to this decision date before fitting
    remaining_time = exercise_times[sabr_exercise_indices[itm]] - exercise_times[k]
    discounted_cashflows = sabr_cashflows[itm] * np.exp(-r * remaining_time)
    coefficients = np.linalg.lstsq(basis, discounted_cashflows, rcond=None)[0]

    # Exercise on paths where the immediate payoff exceeds the fitted value of waiting
    exercise_now = itm[sabr_payoffs[itm, k] > basis @ coefficients]
    sabr_cashflows[exercise_now] = sabr_payoffs[exercise_now, k]
    sabr_exercise_indices[exercise_now] = k

In [33]:
# Discount the chosen cash flows to time zero and compute the standard error.
sabr_pv = sabr_cashflows * np.exp(-r * exercise_times[sabr_exercise_indices])
sabr_price = sabr_pv.mean()
sabr_se = sabr_pv.std(ddof=1) / np.sqrt(N)
print(f"SABR option price: {sabr_price:.4f}")
print(f"Monte Carlo standard error: {sabr_se:.4f} ({N:,} paths)")

SABR option price: 8.5568
Monte Carlo standard error: 0.0109 (1,000,000 paths)
